# ASSIGNMENT 04: CÀI ĐẶT MẠNG NƠ-RON TÍCH CHẬP TỪ ĐẦU BẰNG NUMPY (FROM SCRATCH)
**Học phần:** Phát triển các hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

Mục tiêu notebook:
1. Tự cài đặt phép tích chập Conv1D và Conv2D từ các nguyên lý toán học cơ bản.
2. Tự cài đặt các hàm kích hoạt phi tuyến (ReLU) và tầng gộp cực đại (MaxPooling1D/2D).
3. Triển khai giải thuật lan truyền ngược giải tích (Analytical Backpropagation) cho tầng tích chập.
4. Kiểm thử gradient số học (Numerical Gradient Checking) đối chiếu với giải tích.


In [ ]:
# Cell [1]: Cài đặt lớp Conv1DNumPy (Lan truyền tiến & Lan truyền ngược)
import numpy as np

class Conv1DNumPy:
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        # Khởi tạo He Normalization tránh triệt tiêu gradient
        scale = np.sqrt(2.0 / (in_channels * kernel_size))
        self.W = np.random.randn(out_channels, in_channels, kernel_size) * scale
        self.b = np.zeros((1, out_channels, 1))
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        N, C, L = x.shape
        if self.padding > 0:
            x_pad = np.pad(x, ((0, 0), (0, 0), (self.padding, self.padding)), mode='constant')
        else:
            x_pad = x
        self.x_pad = x_pad
        L_out = (L + 2 * self.padding - self.kernel_size) // self.stride + 1
        out = np.zeros((N, self.out_channels, L_out))
        for i in range(L_out):
            start = i * self.stride
            end = start + self.kernel_size
            x_slice = x_pad[:, :, start:end]
            for oc in range(self.out_channels):
                out[:, oc, i] = np.sum(x_slice * self.W[oc], axis=(1, 2)) + self.b[0, oc, 0]
        return out

    def backward(self, dout, lr=0.01):
        N, C, L = self.x.shape
        L_out = dout.shape[2]
        self.dW = np.zeros_like(self.W)
        self.db = np.sum(dout, axis=(0, 2), keepdims=True)
        dx_pad = np.zeros_like(self.x_pad)

        for i in range(L_out):
            start = i * self.stride
            end = start + self.kernel_size
            for oc in range(self.out_channels):
                dout_val = dout[:, oc, i][:, None, None]
                self.dW[oc] += np.sum(self.x_pad[:, :, start:end] * dout_val, axis=0)
                dx_pad[:, :, start:end] += dout_val * self.W[oc][None, :, :]

        if self.padding > 0:
            dx = dx_pad[:, :, self.padding:-self.padding]
        else:
            dx = dx_pad

        self.W -= lr * self.dW
        self.b -= lr * self.db
        return dx

print("[OK] Đã khởi tạo hoàn tất lớp Conv1DNumPy (Forward & Backward).")


In [ ]:
# Cell [2]: Cài đặt ReLUNumPy và MaxPool1DNumPy
class ReLUNumPy:
    def __init__(self):
        self.mask = None
    def forward(self, x):
        self.mask = (x > 0)
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask

class MaxPool1DNumPy:
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride
        self.x = None
    def forward(self, x):
        self.x = x
        N, C, L = x.shape
        L_out = (L - self.pool_size) // self.stride + 1
        out = np.zeros((N, C, L_out))
        for i in range(L_out):
            start = i * self.stride
            end = start + self.pool_size
            out[:, :, i] = np.max(x[:, :, start:end], axis=2)
        return out

print("[OK] Đã khởi tạo các tầng phụ trợ: ReLUNumPy, MaxPool1DNumPy.")


In [ ]:
# Cell [3]: Kiểm thử Gradient số học đối chiếu với Đạo hàm Giải tích (Gradient Checking)
conv_test = Conv1DNumPy(in_channels=1, out_channels=2, kernel_size=3, padding=1)
x_sample = np.random.randn(2, 1, 8)
out_sample = conv_test.forward(x_sample)
dout_sample = np.ones_like(out_sample)
dx_anal = conv_test.backward(dout_sample, lr=0.0)

# Tính đạo hàm số học cho 1 trọng số W[0, 0, 1]
eps = 1e-6
w_idx = (0, 0, 1)
orig_w = conv_test.W[w_idx]
conv_test.W[w_idx] = orig_w + eps
out_plus = conv_test.forward(x_sample)
conv_test.W[w_idx] = orig_w - eps
out_minus = conv_test.forward(x_sample)
conv_test.W[w_idx] = orig_w

grad_num = np.sum(out_plus - out_minus) / (2 * eps)
grad_anal = conv_test.dW[w_idx]
rel_err = abs(grad_num - grad_anal) / (abs(grad_num) + abs(grad_anal) + 1e-12)

print(f"Gradient Số học (Numerical):  {grad_num:.8f}")
print(f"Gradient Giải tích (Analytical): {grad_anal:.8f}")
print(f"Sai số tương đối (Relative Error): {rel_err:.2e}")
assert rel_err < 1e-7, "Kiểm thử gradient thất bại!"
print("[PASSED] Kiểm thử Gradient giải tích khớp hoàn hảo với xấp xỉ số học!")
